In [1]:
import gymnasium as gym
from dataclasses import dataclass, asdict

import torch as t
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

import numpy as np

In [2]:
env = gym.make('CartPole-v1')

assert isinstance(env.action_space, gym.spaces.Discrete)

n_action = int(env.action_space.n)
obs_dim = env.observation_space.shape

In [4]:
# policy takes in state, outputs action distribution
policy = nn.Sequential(
  nn.Linear(obs_dim[-1], 64),
  nn.ReLU(),
  nn.Linear(64, n_action),
  nn.Softmax(dim=-1)
)

df = 0.9 # discount factor gamma
lr = 3e-3 # this is alpha = step_size
optim = t.optim.Adam(policy.parameters(), lr=lr)

In [5]:
# generate episode

state = t.tensor(env.reset(), dtype=t.float)
done = False

actions, states, rewards = [], [], []
while not done:
  action_probs = policy(state)
  dist = t.distributions.Categorical(probs=action_probs)
  action = dist.sample().item()

  state_next, reward, terminated, truncated, _ = env.step(action)
  done = terminated or truncated
  
  # add to buffer
  actions.append(action)
  states.append(state)
  rewards.append(reward)

  state = t.tensor(state_next, dtype=t.float)


/var/folders/m9/cxt6v3bj4yj_d3cms5m007cm0000gn/T/ipykernel_60719/1050751910.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:255.)
  state = t.tensor(env.reset(), dtype=t.float)


ValueError: expected sequence of length 4 at dim 1 (got 0)

In [ ]:
# compute discounted returns

discounted_returns = [] # G values

for t in range(len(rewards)):
  G = 0.0
  for k, r in enumerate(rewards[t:]):
    G += (df**k)*r
  discounted_returns.append(G)

In [ ]:
# update policy

for state, action, G in zip(states, actions, discounted_returns):
  action_probs = policy(state)
  dist = t.distributions.Categorical(probs=action_probs)
  log_prob = dist.log_prob(action)

  loss = -log_prob*G # we multiply by -1 since gradient descent is designed to decrease loss, but we want to increase the log_prob

  optim.zero_grad()
  loss.backward()
  optim.step()

In [ ]:
# execute policy

state = t.tensor(env.reset(), dtype=t.float)
done = False
env.render()

while not done:
  action_probs = policy(state)
  dist = t.distributions.Categorical(probs=action_probs)
  action = dist.sample().item()

  state_next, reward, done, _ = env.step(action)
  env.render()

  state = t.tensor(state_next, dtype=t.float)
